## Supervised Fine-Tuning (SFT) with Serverless customization on SageMaker AI

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This Immersion Day has been tested using the following SageMaker Distribution images:

<ul>
<li><strong>SageMaker Distribution Image 4.0.3</strong></li>
</ul>  
and the following SageMaker Python SDK version
<ul>
    <li><strong>SageMaker Python SDK version 3.13.1</strong></li>
</ul>
</div>

# Lab 3 – Deploy the Fine-Tuned Model to a Real-Time Endpoint

Lab 2 produced a fine-tuned model and registered it in the Model Registry. In this lab we host that model behind a **SageMaker real-time endpoint** so applications can call it over HTTPS and get low-latency responses.

**The deployment building blocks** (see [Real-time inference](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html) and [Deploy a custom model](https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-trained-model.html)):

1. **EndpointConfig** – declares the instance type, instance count, and routing for the hosting fleet.
2. **Endpoint** – the managed, always-on HTTPS API created from that config.
3. **Model** – points at the merged model artifact in S3 plus the inference container (DJL/LMI) that serves it.
4. **InferenceComponent** – places the model onto the endpoint and reserves the GPU/memory it needs. Inference components let multiple models share one endpoint and scale independently; see [Deploy models with the JumpStartModel class](https://docs.aws.amazon.com/sagemaker/latest/dg/jumpstart-foundation-models-use-python-sdk-model-class.html).

We then send a streaming test prompt and finish by **deleting every resource** so you don't keep paying for an idle GPU endpoint.

***

In [ ]:
%load_ext autoreload
%autoreload 2

### Prerequisites

### Step 1 – Set up the SageMaker session

Establish the SageMaker [`Session`](https://sagemaker.readthedocs.io/en/stable/api/utility/session.html), resolve the **execution role** (the endpoint assumes this role to read the model artifact from S3), and select the default region and bucket.

In [ ]:
import boto3
import os
from rich.pretty import pprint
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

### Step 2 – Locate the fine-tuned model in the Registry

We reconstruct the **Model Package Group** name from `BASE_MODEL_ID` (the same naming scheme Lab 2 used) and select the version to deploy. Change `model_package_version` if you have run training more than once and want an earlier version.

In [ ]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup
from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

model_package_group_name = f"{base_model_id}-sft-mpg"
model_package_version = "1"

model_name = f"{base_model_id}-sft"
endpoint_config_name = f"{base_model_id}-sft-config"
endpoint_name = f"{base_model_id}-sft-endpoint"
ic_name = f"{base_model_id}-sft-ic"

The fine-tuned model package records where the **merged** model artifact lives in S3 (LoRA adapters merged back into the base weights). The cell below resolves that S3 URI — this is exactly the artifact the hosting container will load.

In [ ]:
from sagemaker.core import s3

model_package_group = ModelPackageGroup.get(model_package_group_name)

fine_tuned_model_package_group_arn = model_package_group.model_package_group_arn
print(f"Fine-tuned Model Package Group ARN: {fine_tuned_model_package_group_arn}")

fine_tuned_model_package_arn = f"{model_package_group.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
print(f"Fine-tuned Model Package ARN: {fine_tuned_model_package_arn}")

model_package = ModelPackage.get(fine_tuned_model_package_arn)

# get the merged model artifact and deploy it
merged_model_s3_uri = (
    s3.s3_path_join(
        model_package.inference_specification.containers[
            0
        ].model_data_source.s3_data_source.s3_uri,
        "checkpoints",
        "hf_merged",
    )
    + "/"
)
print(merged_model_s3_uri)

***

### Step 3 – Create the Endpoint Configuration

The **EndpointConfig** is the blueprint for the hosting fleet. The most important choice is the **instance type** — an LLM needs a GPU instance (here `ml.g5.2xlarge`). `health_check_timeout` gives the container enough time to download and load the model before SageMaker marks the endpoint unhealthy, and the routing strategy `LEAST_OUTSTANDING_REQUESTS` sends each request to the least-busy copy.

Set the hosting hardware and timeouts. `ml.g5.2xlarge` provides one NVIDIA A10G GPU, which comfortably serves an 8B-parameter model after quantization.

In [ ]:
instance_count = 1
instance_type = "ml.g5.2xlarge"
health_check_timeout = 700

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig
from sagemaker.core.shapes import ProductionVariant

print(f"Creating EndpointConfig: {endpoint_config_name}")
endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    execution_role_arn=role,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            instance_type=instance_type,
            initial_instance_count=instance_count,
            model_data_download_timeout_in_seconds=health_check_timeout,
            routing_config={"routing_strategy": "LEAST_OUTSTANDING_REQUESTS"},
        )
    ],
)

### Step 4 – Create the Endpoint

A SageMaker **Endpoint** is a fully managed, always-on HTTPS API that hosts your model and serves real-time inference. Creating it provisions the instances from the config above; `wait_for_status("InService")` blocks until the fleet is ready.

> **Security note:** by default this endpoint is invoked through the SageMaker runtime API and is governed by **IAM** — callers must have permission to call `InvokeEndpoint`. There is no separate application-level auth layer, so control access through IAM policies and, if needed, place the endpoint behind [AWS PrivateLink](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-privatelink.html). Don't expose it directly to untrusted clients.

In [ ]:
print(f"Creating Endpoint: {endpoint_name}")
endpoint = Endpoint.create(
    endpoint_name=endpoint_name, endpoint_config_name=endpoint_config_name
)
endpoint.wait_for_status("InService")
print(f"Endpoint {endpoint_name} is InService")

### Step 5 – Define the Model (artifact + serving container)

A SageMaker **Model** binds two things: the **model artifact** in S3 and the **inference container** that loads and serves it. We use the AWS Deep Learning Containers **DJL / LMI** image (Large Model Inference), which runs the model on **vLLM** for high-throughput, streaming LLM serving.

#### Select the inference container image

Build the ECR URI for the DJL/LMI inference image in the current region. The environment variables on the next cell configure how the container serves the model — `OPTION_QUANTIZE=fp8` quantizes weights to fit the GPU, `OPTION_ASYNC_MODE` and the vLLM entrypoint enable streaming, and `OPTION_MAX_MODEL_LEN` caps the context window.

In [ ]:
import json

In [ ]:
region = sess.boto_region_name
CONTAINER_VERSION = "0.36.0-lmi18.0.0-cu128"

inference_image = (
    f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"
)

In [ ]:
env = {
    "HF_MODEL_ID": "/opt/ml/model",  # path to where sagemaker stores the model
    "OPTION_TRUST_REMOTE_CODE": "true",
    "OPTION_MODEL_LOADING_TIMEOUT": "3600",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
    "SERVING_FAIL_FAST": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_DTYPE": "bf16",
    "OPTION_QUANTIZE": "fp8",
    "OPTION_MAX_MODEL_LEN": json.dumps(1024 * 32)
}

In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.resources import TrainingJob
from sagemaker.core.shapes import (
    ContainerDefinition,
    ModelDataSource,
    S3ModelDataSource,
)

fine_tuned_model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        image=inference_image,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=merged_model_s3_uri,
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        ),
        environment=env,
    ),
    execution_role_arn=role,
)

pprint(fine_tuned_model)

### Step 6 – Create the Inference Component

An **InferenceComponent** places the model onto the endpoint and reserves the compute it needs (`number_of_accelerator_devices_required=1`, i.e. one GPU). This decoupling lets several models share a single endpoint and lets you scale or update each model independently. `copy_count=1` runs a single replica for this lab.

In [ ]:
from sagemaker.core.resources import InferenceComponent
from sagemaker.core.shapes import (
    InferenceComponentSpecification,
    InferenceComponentComputeResourceRequirements,
    InferenceComponentRuntimeConfig,
)

# Step 3: Create InferenceComponent
inference_component = InferenceComponent.create(
    inference_component_name=ic_name,
    endpoint_name=endpoint_name,
    variant_name="AllTraffic",
    specification=InferenceComponentSpecification(
        model_name=model_name,
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            min_memory_required_in_mb=10240,
            number_of_accelerator_devices_required=1,
        ),
    ),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1),
    region=region,
)

print(f"InferenceComponent created: {inference_component.inference_component_name}")
print(f"Endpoint ARN: {endpoint.endpoint_arn}")
inference_component.wait_for_status("InService")
print(f"Endpoint {ic_name} is InService")

***

### Step 7 – Test the endpoint

We send a medical-reasoning prompt and **stream** the response token-by-token using `invoke_endpoint_with_response_stream`. Streaming improves perceived latency — you see output as it is generated instead of waiting for the full completion. Watch for the `<think>` reasoning block in the output: that style is exactly what the fine-tuning in Lab 1/2 taught the model to produce.

In [ ]:
import io
import json
import boto3

In [ ]:
sagemaker_client = boto3.client(service_name="sagemaker-runtime")

#### Helper: parse the streaming response

The runtime returns a stream of byte chunks. `LineIterator` buffers those chunks and yields one complete line at a time so we can decode each server-sent event as it arrives.

In [ ]:
class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()

            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]

            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise

            if "PayloadPart" not in chunk:
                continue

            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

Utility function to parse model answer

In [ ]:
def parse_streaming_response(line_str):
    """Parse a single streaming response line and return content if found."""
    if not line_str.strip() or line_str.strip() == "data: [DONE]":
        return None

    if line_str.startswith("data: "):
        line_str = line_str[6:]

    try:
        data = json.loads(line_str)
        if "choices" in data:
            for choice in data["choices"]:
                if "delta" in choice and "content" in choice["delta"]:
                    return choice["delta"]["content"]
    except json.JSONDecodeError:
        pass

    return None

In [ ]:
prompt = """
Regarding the temporomandibular joint, which statements are true or false: 
Is the temporomandibular joint a synovial joint? 
Is the articular disc a remnant of the tendon of the medial pterygoid? 
Do gliding movements occur in the lower compartment and rotatory movements occur in the upper compartment? 
Is the joint capsule thick and tight in the lower part and loose and lax in the upper part? 
Does the sphenomandibular ligament act as a false support to the joint and attach to the angle of the mandible?
"""

In [ ]:
request_body = {
    "model_name": ic_name,
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ],
    "max_tokens": 4096,
    "temperature": 0.3,
    "top_p": 0.9,
    "stop": ["<|im_end|>"],
    "stream": True,
}

response = sagemaker_client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    Body=json.dumps(request_body),
    ContentType="application/json",
)

generated_text = ""

for line in LineIterator(response["Body"]):
    if line:
        content = parse_streaming_response(line.decode("utf-8"))
        if content:
            generated_text += content
            print(content, end="", flush=True)

***

### Step 8 – Clean up

**A real-time endpoint bills for its instances as long as it exists**, whether or not it is receiving traffic. Delete the resources in reverse order of creation — inference component, model, endpoint, then endpoint config — to stop charges. See [Delete Endpoints and Resources](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-delete-resources.html).

In [ ]:
from sagemaker.core.resources import InferenceComponent

# Delete inference component
InferenceComponent.get(inference_component_name=ic_name).delete()

In [ ]:
from sagemaker.core.resources import Model

# Delete model
Model.get(model_name=model_name).delete()

In [ ]:
from sagemaker.core.resources import Endpoint

# Delete endpoint (optional - if you want to remove the endpoint too)
Endpoint.get(endpoint_name=endpoint_name).delete()

In [ ]:
from sagemaker.core.resources import EndpointConfig

# Delete endpoint config (optional)
EndpointConfig.get(endpoint_config_name=endpoint_config_name).delete()